# LABORATORIO CALIFICADO N° 05
## Fundamentos de Gestión de Datos — Semana 10
**Docente:** Pilar Rocío Sayán Mejía | **Semestre:** 2026-I

---
## CASO: La Migración de LogiPeru
**Protagonista:** Roberto Vega, Data Engineer Senior
**Empresa:** LogiPeru S.A. — Empresa de logística con 8 almacenes en Perú

**3 tareas del gerente de operaciones:**
1. Segmentar clientes con K-Means para campañas diferenciadas
2. Migrar datos del sistema legacy a SQLite via ETL
3. Demostrar transacciones seguras con ACID

---

## PASO 1: Generación de Datos Sintéticos de LogiPeru

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

np.random.seed(42)
n=200

ciudades=['Lima','Arequipa','Trujillo','Cusco','Piura','Iquitos','Tacna','Chiclayo']
# Grupos naturales: clientes frecuentes premium, regulares, ocasionales
grupos=np.random.choice([0,1,2],n,p=[0.3,0.4,0.3])

freq_envio=np.where(grupos==0,np.random.randint(15,30,n),
           np.where(grupos==1,np.random.randint(5,15,n),np.random.randint(1,5,n))).astype(float)
peso_prom=np.where(grupos==0,np.random.uniform(20,50,n),
          np.where(grupos==1,np.random.uniform(8,20,n),np.random.uniform(1,8,n)))
valor_envio=freq_envio*peso_prom*np.random.uniform(0.8,1.2,n)*5

# Outliers intencionales
for idx in np.random.choice(n,5,replace=False):
    valor_envio[idx]=valor_envio[idx]*8

df=pd.DataFrame({
    'id_cliente':[f'LP{i:04d}' for i in range(1,n+1)],
    'ciudad':np.random.choice(ciudades,n),
    'freq_envios_mes':freq_envio.round(0).astype(int),
    'peso_prom_kg':peso_prom.round(2),
    'valor_envio_total':valor_envio.round(2),
    'anos_cliente':np.random.randint(1,10,n),
    'segmento_real':['Premium' if g==0 else 'Regular' if g==1 else 'Basico' for g in grupos],
})
print(f'Dataset LogiPeru: {df.shape}')
display(df.head())

## PASO 2: Clustering K-Means — Segmentación de Clientes

In [ ]:
# Variables para clustering
X_cols=['freq_envios_mes','peso_prom_kg','valor_envio_total','anos_cliente']
X=df[X_cols].values

# Normalizar
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

# Metodo del codo
inertias=[]
silhouettes=[]
k_range=range(2,9)
for k in k_range:
    km=KMeans(n_clusters=k,random_state=42,n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled,km.labels_))

fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(k_range,inertias,'bo-',linewidth=2,markersize=8)
axes[0].set_xlabel('Numero de Clusters K'); axes[0].set_ylabel('Inercia')
axes[0].set_title('Metodo del Codo')
axes[1].plot(k_range,silhouettes,'ro-',linewidth=2,markersize=8)
axes[1].set_xlabel('Numero de Clusters K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score por K')
plt.tight_layout(); plt.show()

K_optimo=silhouettes.index(max(silhouettes))+2
print(f'K optimo segun silhouette: {K_optimo} (score={max(silhouettes):.3f})')

## PASO 3: Interpretación de Segmentos de Clientes

In [ ]:
km_final=KMeans(n_clusters=K_optimo,random_state=42,n_init=10)
df['cluster']=km_final.fit_predict(X_scaled)

# Visualizacion 2D
fig,ax=plt.subplots(figsize=(9,6))
colors=['#2E75B6','#1A7F37','#C00000','#ED7D31']
for k in range(K_optimo):
    mask=df['cluster']==k
    ax.scatter(df.loc[mask,'freq_envios_mes'],df.loc[mask,'valor_envio_total'],
               c=colors[k],label=f'Cluster {k}',alpha=0.7,s=60)
ax.set_xlabel('Frecuencia Envios/Mes'); ax.set_ylabel('Valor Total Envios (S/.)')
ax.set_title(f'Segmentacion de Clientes LogiPeru (K={K_optimo})')
ax.legend(); plt.tight_layout(); plt.show()

# Perfil de clusters
print('PERFIL DE CADA CLUSTER:')
perfil=df.groupby('cluster')[X_cols].mean().round(2)
display(perfil)

# Nombrar clusters (ajustar segun resultados)
nombres_clusters={}
for k in range(K_optimo):
    freq=perfil.loc[k,'freq_envios_mes']; val=perfil.loc[k,'valor_envio_total']
    if freq>12 or val>1000:
        nombres_clusters[k]='Cliente Frecuente Premium'
    elif freq>5:
        nombres_clusters[k]='Cliente Regular'
    else:
        nombres_clusters[k]='Cliente Ocasional Basico'
df['nombre_segmento']=df['cluster'].map(nombres_clusters)
print('\nDistribucion por segmento:')
print(df['nombre_segmento'].value_counts())

## PASO 4: Transacciones DML y ACID

In [ ]:
conn=sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON')

# Crear tabla
conn.execute('''
CREATE TABLE clientes_logiperu (
    id_cliente TEXT PRIMARY KEY,
    ciudad TEXT,
    freq_envios_mes INTEGER,
    valor_envio_total REAL,
    segmento TEXT
)
''')

# INSERT con transaccion
print('INSERT: cargando 10 clientes de prueba...')
for i,row in df.head(10).iterrows():
    conn.execute('INSERT INTO clientes_logiperu VALUES (?,?,?,?,?)',
                 (row['id_cliente'],row['ciudad'],row['freq_envios_mes'],
                  row['valor_envio_total'],row['nombre_segmento']))
conn.commit()
print('COMMIT: 10 registros confirmados')

# UPDATE
conn.execute("UPDATE clientes_logiperu SET segmento='VIP' WHERE freq_envios_mes > 20")
conn.commit()
print('UPDATE: segmento VIP aplicado a clientes frecuentes')

# ROLLBACK demo
print('\nDEMO ROLLBACK:')
conn.execute("UPDATE clientes_logiperu SET valor_envio_total = 0")
print('Antes del ROLLBACK:', pd.read_sql_query('SELECT AVG(valor_envio_total) as promedio FROM clientes_logiperu',conn).iloc[0,0])
conn.rollback()
print('Despues del ROLLBACK:', pd.read_sql_query('SELECT AVG(valor_envio_total) as promedio FROM clientes_logiperu',conn).iloc[0,0])
print('Valores restaurados correctamente.')

## PASO 5 y 6: Pipeline ETL Completo + Validación

In [ ]:
print('='*50)
print('PIPELINE ETL: LOGIPERU LEGACY -> SQLITE')
print('='*50)

# EXTRACT: simular lectura del sistema legacy (CSV en memoria)
print('\n[EXTRACT] Leyendo sistema legacy...')
import io
csv_legacy=df.to_csv(index=False)
df_raw=pd.read_csv(io.StringIO(csv_legacy))
print(f'  Registros extraidos: {len(df_raw)}')

# TRANSFORM
print('\n[TRANSFORM] Aplicando transformaciones...')
df_transform=df_raw.copy()
df_transform['ciudad']=df_transform['ciudad'].str.title()
df_transform['freq_envios_mes']=df_transform['freq_envios_mes'].clip(0,50)
df_transform['valor_envio_total']=df_transform['valor_envio_total'].round(2)
n_dups=df_transform.duplicated(subset=['id_cliente']).sum()
df_transform=df_transform.drop_duplicates(subset=['id_cliente'])
print(f'  Duplicados eliminados: {n_dups}')
print(f'  Registros tras limpieza: {len(df_transform)}')

# LOAD
print('\n[LOAD] Cargando en SQLite...')
conn_etl=sqlite3.connect(':memory:')
df_transform.to_sql('clientes_migrados',conn_etl,if_exists='replace',index=False)
n_cargados=pd.read_sql_query('SELECT COUNT(*) as n FROM clientes_migrados',conn_etl).iloc[0,0]
print(f'  Registros cargados en SQLite: {n_cargados}')

# Reporte de calidad post-ETL
print('\n[REPORTE CALIDAD POST-ETL]')
print(f'  Origen:   {len(df_raw)} registros')
print(f'  Destino:  {n_cargados} registros')
print(f'  Integridad: {round(n_cargados/len(df_raw)*100,1)}%')
display(pd.read_sql_query('SELECT segmento_real, COUNT(*) as total FROM clientes_migrados GROUP BY segmento_real',conn_etl))

## ACTIVIDAD 3: Análisis Reflexivo

**A.** ¿Con cuántos clusters terminó el análisis? Justifique usando el silhouette score.

*(Su respuesta aqui)*

---

**B.** ¿Qué estrategia comercial propone para el cluster con mayor valor de envío?

*(Su respuesta aqui)*

---

**C.** ¿Por qué es importante el ROLLBACK en la migración de datos de LogiPeru?

*(Su respuesta aqui)*

---

**D.** ¿Qué problemas ocurrirían si el ETL no valida duplicados antes de cargar?

*(Su respuesta aqui)*

## CONCLUSIONES

1. *(Conclusión 1)*

2. *(Conclusión 2)*

3. *(Conclusión 3)*

---
**Docente:** Pilar Rocío Sayán Mejía | **FGD 2026-I** | **LAB-C5**